<a href="https://colab.research.google.com/github/ID26S422/The-Feature-Matching-Project/blob/main/SIFT_Matchig.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from PIL import Image
import os

In [ ]:
#Load Images
image1_path="images/Image 1.jpg"
image2_path="images/Image 2.jpg"
image3_path="images/Image 3.jpg"

img1=np.array(Image.open(image1_path))
img2=np.array(Image.open(image2_path))
img3=np.array(Image.open(image3_path))

print("image 1 size:", img1.shape)
print("image 1 dtype:", img1.dtype)

print("\n")
print("image 2 size:", img2.shape)
print("image 2 dtype:", img2.dtype)

print("\n")
print("image 3 size:", img3.shape)
print("image 3 dtype:", img3.dtype)

plt.figure(figsize=(30,10))

plt.subplot(1,3,1)
plt.imshow(img1)
plt.title("Image 1", fontsize=20)
plt.axis('off')

plt.subplot(1,3,2)
plt.imshow(img2)
plt.title("Image 2", fontsize=20)
plt.axis('off')

plt.subplot(1,3,3)
plt.imshow(img3)
plt.title("Image 3", fontsize=20)
plt.axis('off')

plt.show()

In [ ]:
os.makedirs("results/sift", exist_ok=True)

In [ ]:
#RGBtoGRAY
img1_gray=cv2.cvtColor(img1,cv2.COLOR_RGB2GRAY)
img2_gray=cv2.cvtColor(img2,cv2.COLOR_RGB2GRAY)
img3_gray=cv2.cvtColor(img3,cv2.COLOR_RGB2GRAY)

print("image 1 in gray, shape:", img1_gray.shape)
print("image 2 in gray, shape:", img2_gray.shape)
print("image 3 in gray, shape:", img3_gray.shape)

#save gray images to results
cv2.imwrite("results/sift/Image 1 Gray.png",img1_gray)
cv2.imwrite("results/sift/Image 2 Gray.png",img2_gray)
cv2.imwrite("results/sift/Image 3 Gray.png",img3_gray)

#plotting gray images
plt.figure(figsize=(30,10))

plt.subplot(1,3,1)
plt.imshow(img1_gray, cmap='gray')
plt.title("Image 1 in gray", fontsize='20')
plt.axis('off')

plt.subplot(1,3,2)
plt.imshow(img2_gray, cmap='gray')
plt.title("Image 2 in gray", fontsize='20')
plt.axis('off')

plt.subplot(1,3,3)
plt.imshow(img3_gray, cmap='gray')
plt.title("Image 3 in gray", fontsize='20')
plt.axis('off')

plt.show()

In [ ]:
#sift object creation
sift = cv2.SIFT_create()

#finding keypoints
keypoints1 = sift.detect(img1_gray, None)
keypoints2 = sift.detect(img2_gray, None)
keypoints3 = sift.detect(img3_gray, None)

#drawing keypoints on images
img1_keypoints = cv2.drawKeypoints(
    img1_gray,
    keypoints1,
    None,
    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
)

img2_keypoints = cv2.drawKeypoints(
    img2_gray,
    keypoints2,
    None,
    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
)

img3_keypoints = cv2.drawKeypoints(
    img3_gray,
    keypoints3,
    None,
    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
)

#plotting the keypoints on image
plt.figure(figsize=(30, 10))

plt.subplot(1, 3, 1)
plt.imshow(img1_keypoints, cmap="gray")
plt.title(f"Image 1: {len(keypoints1)} keypoints", fontsize='20')
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(img2_keypoints, cmap="gray")
plt.title(f"Image 2: {len(keypoints2)} keypoints", fontsize='20')
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(img3_keypoints, cmap="gray")
plt.title(f"Image 3: {len(keypoints3)} keypoints", fontsize='20')
plt.axis("off")

plt.show()

#saving keypoint imprinted images in results folder
cv2.imwrite("results/sift/Image 1 Keypoints.png", img1_keypoints)
cv2.imwrite("results/sift/Image 2 Keypoints.png", img2_keypoints)
cv2.imwrite("results/sift/Image 3 Keypoints.png", img3_keypoints)

In [ ]:
#Find Descriptors of Keypoints
keypoints1, descriptors1 = sift.compute(img1_gray, keypoints1)
keypoints2, descriptors2 = sift.compute(img2_gray, keypoints2)
keypoints3, descriptors3 = sift.compute(img3_gray, keypoints3)

print("Image 1")
print("Number of keypoints:", len(keypoints1))
print("Descriptor shape:", descriptors1.shape)

print("\nImage 2")
print("Number of keypoints:", len(keypoints2))
print("Descriptor shape:", descriptors2.shape)

print("\nImage 3")
print("Number of keypoints:", len(keypoints3))
print("Descriptor shape:", descriptors3.shape)

In [ ]:
#descriptor example: a one line array
print(descriptors1[0])

#descriptor array visualizing the 8 bins of each 16 cells
print("\nDescriptor for Keypoint 0:")
print(descriptors1[0].reshape(16, 8))
print("\n")

#descriptor visualize in a graph:
plt.figure(figsize=(10, 6))
plt.imshow(descriptors1[0].reshape(16, 8),
           cmap="gray",
           aspect="auto"
           )
plt.colorbar(label="Descriptor value")
plt.xlabel("Orientation bins")
plt.ylabel("Spatial cells")
plt.title("SIFT Descriptor for Keypoint 0")

#save one descriptor image for understanding as png in results folder
plt.savefig(
    "results/sift/Image 1 Descriptor Keypoint 0.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
#BF matchig object creation
bf = cv2.BFMatcher()

#do BF(brute force) matchig and list 2 best matches:
matches12 = bf.knnMatch(descriptors1, descriptors2, k=2)

print("shape of matches12", len(matches12))  #should be equal to no of keypoints
print(matches12[0])                          #should give first 2 best matches for keypoint 0 of image 1

#find valid matches using a basic lowes ratio formula. keeping ratio at 0.75.
good_matches12 = []

for m1,m2 in matches12:
  if m1.distance/m2.distance<0.75:
    good_matches12.append(m1)

print(len(good_matches12)) #gives how many matches are actually valid.

In [ ]:
matches23 = bf.knnMatch(descriptors2, descriptors3, k=2)

print("shape of matches12", len(matches23))
print(matches23[0])

good_matches23 = []

for m3,m4 in matches23:
  if m3.distance/m4.distance<0.75:
    good_matches23.append(m3)

print(len(good_matches23))

In [ ]:
#plotting matches on images
img_matches12 = cv2.drawMatches(
    img1_gray,
    keypoints1,
    img2_gray,
    keypoints2,
    good_matches12,
    None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
)

img_matches23 = cv2.drawMatches(
    img2_gray,
    keypoints2,
    img3_gray,
    keypoints3,
    good_matches23,
    None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
)

plt.figure(figsize=(20,10))

plt.subplot(2,1,1)
plt.imshow(img_matches12, cmap="gray")
plt.title("Matches: Image 1 ↔ Image 2")
plt.axis("off")

plt.subplot(2,1,2)
plt.imshow(img_matches23, cmap="gray")
plt.title("Matches: Image 2 ↔ Image 3")
plt.axis("off")

plt.show()

#saving match imprinted images in results folder
cv2.imwrite("results/sift/Image 1-2 Matches.png", img_matches12)
cv2.imwrite("results/sift/Image 2-3 Matches.png", img_matches23)